In [1]:
import matplotlib
import platform
print ("Operating system: ", platform.system())
if "Linux" in platform.system():
    %matplotlib tk
else:
    %matplotlib qt
    

import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
import scipy
import os
import time

#
from calibration.calibration import BMICalibration
from drift.drift import make_template, compute_drift_multi_frames, correct_drift, correct_drift_single_frame
from utils.utils import smooth_ca_time_series, compute_dff0



Operating system:  Linux


Autosaving every 180 seconds


/home/cat/anaconda3/envs/bmi2/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#
def template_generation(bmi_c,
                        ):
    n_iter = 1
    bmi_c.n_imgs_to_sample = 1000
    random_img_sample_flag = False
    bmi_c.n_best_imgs = 100
    plotting = True
    n_cores = 8
    template=None
    idx_imgs = None
    for k in range(n_iter):
        #idx_imgs = None
        corr_maxs, template, idx_imgs = make_template(
                                                  bmi_c.data, 
                                                  bmi_c.fname,
                                                  bmi_c.n_imgs_to_sample,
                                                  bmi_c.n_best_imgs,
                                                  template,
                                                  idx_imgs,
                                                  random_img_sample_flag,
                                                  plotting,
                                                  n_cores)
    bmi_c.template = template
    bmi_c.idx_imgs = idx_imgs
    
    #
    return bmi_c

#
def plot_mean_vs_template(bmi_c):
    
    plt.figure()
    print (bmi_c.data.shape, bmi_c.idx_imgs.shape)
    temp = bmi_c.data[bmi_c.idx_imgs].mean(0)
    plt.title("Average map over " + str(bmi_c.idx_imgs.shape[0]) + " images")
    plt.imshow(temp,
               #vmin=0,
               #vmax=1500
               #
               )

    #
    plt.figure()
    plt.title("Average map over highest correlated " + str(bmi_c.n_best_imgs) + " images")
    plt.imshow(bmi_c.template,
               #vmin=0,
               #vmax=1500
               )

    #
    plt.show()
    

In [3]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################
#fname = '/media/cat/4TB/donato/DON-008498/22-06-08/calibration/Image_001_001.raw'
#fname = '/home/cat/data/donato/bscope_tests/8848/data/Image_001_001.raw'
#fname = '/home/cat/data/donato/bscope_tests/8848/data/Image_001_001_original.raw'
#fname = '/home/cat/data/donato/bscope_tests/8499/data/Image_001_001.raw'
fname = r'D:\bmi\DON10775\22-07-19\calibration\Image_001_001.raw'
fname = '/media/cat/4TB/donato/DON-009460/22-06-21/calibration/Image_001_001.raw'
fname = '/media/cat/4TB/donato/DON-008499/mouse2_calibration/Image_001_001.raw'
fname = '/home/cat/data/donato/bscope_tests/8499/data/Image_001_001_original_do_not_overwrite.raw'
# 
bmi_c = BMICalibration(fname)

bmi_c.smooth_ca_time_series = smooth_ca_time_series
bmi_c.compute_dff0 = compute_dff0

#
bmi_c.subsample = 10 # for std computation downsample to every N'th frame; the more frames the better the rois;
                  #   TODO: use correlation instead?! might be much faster; it is quit fast in other implemenations


memmap :  (10000, 512, 512)


In [4]:
####################################################################
########################## FIND TEMPLATE ###########################
####################################################################

#
n_iterations = 1  # number of times to iterate over the template
n_cores = 8
subsample = 3  # subsample every N'th frame for drift to speed up;
                # TODO: would be good to use all frames!!

for k in range(n_iterations):
    # make template
    bmi_c = template_generation(bmi_c)    

    # detection of motion
    shifts, _ = compute_drift_multi_frames(k,
                                           bmi_c,
                                           subsample,
                                           n_cores,)

    # correct motion (+ save to disk)
    bmi_c = correct_drift(k,
                          bmi_c, 
                          shifts)

# remake template
bmi_c = template_generation(bmi_c)    

#
plot_mean_vs_template(bmi_c)
    
print ("DONE...")

  0%|          | 0/8 [00:00<?, ?it/s]

computing motion on:  /media/cat/4TB/donato/DON-009460/22-06-21/calibration/Image_001_001.raw


  0%|          | 0/30 [00:00<?, ?it/s]

phase corr computation: 100%|██████████| 223/223 [01:09<00:00,  3.19it/s]


TODO: undo interpolation for drift with better function


fixing drift calibration data: 100%|██████████| 20000/20000 [00:04<00:00, 4151.25it/s]


  0%|          | 0/8 [00:00<?, ?it/s]

(20000, 512, 512) (1000,)
DONE...


In [4]:
############################################################
########### COMPUTE STD/MEAN and MAX PROJ MAP ##############
############################################################

bmi_c.data.shape
idx = np.random.choice(np.arange(bmi_c.data.shape[0]),
                       2000, replace=False)
                       
temp = bmi_c.data[idx]

maxproj = np.max(temp,axis=0)

std = np.std(temp,axis=0)
#mean = np.mean(temp,axis=0)



In [22]:
from matplotlib.widgets import Slider, Button, RadioButtons
from scipy.ndimage import gaussian_filter

fig = plt.figure()
ax=plt.subplot(111)
live_image_vmin = 0
live_image_vmax = 25000

image_obj = ax.imshow(maxproj,
          vmin=live_image_vmin,
          vmax=live_image_vmax
          )

axmin = fig.add_axes([0.55, 0.90, 0.1, 0.03])
axmax  = fig.add_axes([0.55, 0.93, 0.1, 0.03])

#
smin = Slider(axmin, 'Min', 0, live_image_vmax, valinit=live_image_vmin)
smax = Slider(axmax, 'Max', 0, live_image_vmax, valinit=live_image_vmax)

#
def update_clim(val):
    image_obj.set_clim([smin.val,
                        smax.val])
    res = gaussian_filter(maxproj, sigma=1)
    image_obj.set_data(res)
    
def update_n_frames_average(val):
    live_image_average_n_frames = int(n_frame_ave.val)

#
smin.on_changed(update_clim)
smax.on_changed(update_clim)

#
plt.show()

Exception in Tkinter callback
Traceback (most recent call last):
  File "/home/cat/anaconda3/envs/bmi2/lib/python3.8/tkinter/__init__.py", line 1892, in __call__
    return self.func(*args)
  File "/home/cat/anaconda3/envs/bmi2/lib/python3.8/tkinter/__init__.py", line 814, in callit
    func(*args)
  File "/home/cat/anaconda3/envs/bmi2/lib/python3.8/site-packages/matplotlib/backends/_backend_tk.py", line 252, in idle_draw
    self.draw()
  File "/home/cat/anaconda3/envs/bmi2/lib/python3.8/site-packages/matplotlib/backends/backend_tkagg.py", line 9, in draw
    super().draw()
  File "/home/cat/anaconda3/envs/bmi2/lib/python3.8/site-packages/matplotlib/backends/backend_agg.py", line 436, in draw
    self.figure.draw(self.renderer)
  File "/home/cat/anaconda3/envs/bmi2/lib/python3.8/site-packages/matplotlib/artist.py", line 73, in draw_wrapper
    result = draw(artist, renderer, *args, **kwargs)
  File "/home/cat/anaconda3/envs/bmi2/lib/python3.8/site-packages/matplotlib/artist.py", line 

In [51]:
from matplotlib.widgets import Slider, Button, RadioButtons

#


sigma = 0.5

#
fig = plt.figure()

#
ax=plt.subplot(121)
live_image_vmin = 0
live_image_vmax = 3000

std = std_map.copy()
#
image_obj = plt.imshow(std,
          vmin=live_image_vmin,
          vmax=live_image_vmax
          )

axmin = fig.add_axes([0.05, 0.90, 0.1, 0.03])
axmax  = fig.add_axes([0.05, 0.93, 0.1, 0.03])

#
smin = Slider(axmin, 'Min', 0, live_image_vmax, valinit=live_image_vmin)
smax = Slider(axmax, 'Max', 0, live_image_vmax, valinit=live_image_vmax)

#
def update_clim1(val):
    image_obj.set_clim([smin.val,
                        smax.val])
    res = gaussian_filter(std, sigma=sigma)
    image_obj.set_data(res)
    
#
smin.on_changed(update_clim1)
smax.on_changed(update_clim1)

####################################################
ax=plt.subplot(122)
live_image_vmin2 = 0
live_image_vmax2 = 3000

#
image_obj2 = plt.imshow(std,
          vmin=live_image_vmin2,
          vmax=live_image_vmax2
          )

axmin2 = fig.add_axes([0.55, 0.90, 0.1, 0.03])
axmax2  = fig.add_axes([0.55, 0.93, 0.1, 0.03])

#
smin2 = Slider(axmin2, 'Min', 0, live_image_vmax2, valinit=live_image_vmin2)
smax2 = Slider(axmax2, 'Max', 0, live_image_vmax2, valinit=live_image_vmax2)

#
def update_clim2(val):
    image_obj2.set_clim([smin2.val,
                        smax2.val])
    res = gaussian_filter(std, sigma=sigma)
    image_obj2.set_data(res)
    
#
smin2.on_changed(update_clim2)
smax2.on_changed(update_clim2)


plt.show(block=True)

print ("max proj values: ", smin.val, smax.val)
print ("std: ", smin2.val, smax2.val)

img_std = std.copy()
idx = np.where(img_std<smin2.val)
idx2 = np.where(img_std>=smin2.val)

img_std[idx] = 0
img_std[idx2] = 1
img_std = gaussian_filter(img_std, sigma=sigma)

#
img_maxproj = std.copy()
idx = np.where(img_maxproj<smin.val)
idx2 = np.where(img_maxproj>=smin.val)

img_maxproj[idx] = 0
img_maxproj[idx2] = 1
img_maxproj = gaussian_filter(img_maxproj, sigma=sigma)

    

max proj values:  0 3000
std:  190.14084507042207 206.39219934994617


In [32]:
from stardist.models import StarDist2D
from stardist.data import test_image_nuclei_2d
from stardist.plot import render_label
from csbdeep.utils import normalize
import matplotlib.pyplot as plt

# prints a list of available models
print (StarDist2D.from_pretrained())

# creates a pretrained model
model = StarDist2D.from_pretrained('2D_versatile_fluo')



There are 4 registered models for 'StarDist2D':

Name                  Alias(es)
────                  ─────────
'2D_versatile_fluo'   'Versatile (fluorescent nuclei)'
'2D_versatile_he'     'Versatile (H&E nuclei)'
'2D_paper_dsb2018'    'DSB 2018 (from StarDist 2D paper)'
'2D_demo'             None
None
Found model '2D_versatile_fluo' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479071, nms_thresh=0.3.


In [52]:
#
img = img_std #test_image_nuclei_2d()
#img = img_maxproj #test_image_nuclei_2d()

#img = normalize(img[16], 1,99.8, axis=axis_norm)
labels, details = model.predict_instances(img)

plt.figure(figsize=(8,8))
plt.imshow(img if img.ndim==2 else img[...,0], clim=(0,1), cmap='gray')
plt.imshow(labels, cmap='viridis', alpha=0.5)
plt.axis('off');


1/1 [==============================] - 0s 397ms/step


In [41]:
#######################################################################
########### COMPUTE STD OVER TIME TO GET CELL FOOTPRINTS ##############
#######################################################################
# TODO: the gaussian filter step takes a long time
#      - try to speed up
#      - also, corelation map might be even better method for detecting ROIs
#         e.g. see suite2p's visualization tool <- seems pretty quick

# Also, need to exponse some more variables here like imshow vmin/vmax which are used to plot the final result
start = time.time()
std_map = bmi_c.make_std_map()
#std_map = bmi_c.make_max_proj_map()
print ("total processing time: ", time.time()-start, " sec")
print ("...DONE...")


data into analysis:  (1000, 512, 512)
 gaussian filter width:  1 , order:  0
done filtering... (TO CHECK which axis are we filtering!!)
total processing time:  21.4890775680542  sec
...DONE...


In [40]:
#################################################################################
########### FIND CORRECT VMIN VMAX FOR MAXIMUM WATERSHED DETECTION ##############
#################################################################################
bmi_c.vmin = 15000
bmi_c.vmax = 65000

#
#bmi_c.vmin = 100
#bmi_c.vmax = 550

#std_map2 = template
# get a thrsholded map which hides most of the noise
std_map2 = bmi_c.plot_std_map(std_map)
print ("DONE")

staring computing std...
done computing std...
DONE


In [42]:
#######################################################################
########### RUN WATERSHED ALGORITHM DETECTION ON STD MAP ##############
#######################################################################

bmi_c.min_size_roi = 75
bmi_c.max_size_roi = 700
bmi_c.sigma = 0.1
bmi_c.n_smooth_steps = 1
bmi_c.find_roi_boundaries(std_map2)

#
bmi_c.scale=3000                      # spacing between each neuron trace because they are not normalized to the max vlaue
bmi_c.trace_subsample = 10             # Subsample the time series to go faster;

# visualize traces
bmi_c.compute_traces2(std_map2)

looping over cells: 100%|██████████| 929/929 [00:00<00:00, 1171.02it/s]


plotting cells:  [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55]
memmap :  (20000, 512, 512)


100%|██████████| 2000/2000 [00:05<00:00, 355.44it/s]


[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55]


In [15]:
###############################################################
########### SELECT CELLS TO BE USED FOR ENSEMBLES #############
###############################################################

# save ensemble rois
bmi_c.ensemble1 = [1,2]
bmi_c.ensemble2 = [58,54]
both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
print ("all cells:", both)

#
bmi_c.show_traces_ids(both)


all cells: [ 1  2 58 54]


### COMPUTE THE MIN AND MAXES FOR THE SELECTED ENSMEMLES

Some important points:

1. For now we are working in pixel absolute values for each cell.  

2. A better option might be to find the maximum peak of a cell during a window and then save that value and normalize all future events by that value. (note: any online BMI filtering/chagnig of data will need to account for this).


In [16]:
# ######################################################################
# ########### RECOMPUTE TRACES WITH SINGLE FRAME PRECISION #############
# ######################################################################
bmi_c.trace_subsample = 1        # Subsample the time series to go faster;

# visualize traces
bmi_c.compute_traces2(std_map2, both)

print ("DONE...")

plotting cells:  [ 1  2 58 54]
memmap :  (20000, 512, 512)


100%|██████████| 20000/20000 [00:04<00:00, 4074.56it/s]


[ 1  2 58 54]
DONE...


In [17]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################
# 
bmi_c.sample_rate = 30
bmi_c.post_reward_lockout = 10   # reward lockout in seconds
bmi_c.balance_ensemble_rewards_flag = True   #this makes sure that both ensembles elicit a similar number of random rewards
bmi_c.rois_smooth_window = 10     # of frames to use to smooth the realtime signal
bmi_c.smooth_diff_function_flag = True    # use a kernel window to smooth current value

# find 30% reward threshold
# We have 3 options: 
#   1) rewarding  E1-E2 going above some threshold
#   2) rewarding E2-E1 going above some threhosld
#   3) rewarding both
# The challenge is mapping the ensemble states to tone/stimulus space
# 
#gr.find_reward_thresholds_low_and_high()
#gr.find_reward_thresholds_high()  # this only rewards when sound passes specific level

# this option rewards both ensembles 
normalize_peaks = True   # this flag normalizes the peaks to make sure one ensembel
                         # doesn't completely dominate the other
bmi_c.find_reward_thresholds_absolute(normalize_peaks)


#
print ("thresholds: ", bmi_c.high)

########################################
bmi_c.plot_rewarded_ensembles()

print (bmi_c.high)
bmi_c.high = bmi_c.high*1

100%|██████████| 19990/19990 [00:00<00:00, 52972.24it/s]


TODO: Normalize the peaks of the 2 Ensembles so the mice don't learn to use one esnemble against the other!!!!
low, high:  nan 5.647784066013194
nsec recording:  666 max # of random rewards (i.e. every 30sec)  22 # of rewards for 30% of the time:  6
updated rwards #:  7 nan 2.243380698580615
thresholds:  2.243380698580615
2.243380698580615


In [18]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################

# save all data to disk
# also add the tone values here as well that will be used for the experiment
bmi_c.low_freq = 2000
bmi_c.high_freq = 18000

# save cell pixel footprints locations as 2 column array inside list
cells = []
for k in range(4):
    temp = bmi_c.indexes[both[k]]
    temp1 = temp[0]
    temp2 = temp[1]
    temp = np.vstack((temp1,temp2))
    cells.append(temp)

# also grab contours of cells; both contains all cell ids
contours = bmi_c.compute_contour_map(std_map, both)
print ("len: ", len(contours))        

# save individual pixels of each cell - currently implemented in BMI
np.savez(os.path.join(os.path.split(os.path.split(fname)[0])[0],
                        'rois_pixels_and_thresholds.npz'),
            cell0_footprint = cells[0],
            cell1_footprint = cells[1],
            cell2_footprint = cells[2],
            cell3_footprint = cells[3],
            
            #
            cell0_contour = contours[0],
            cell1_contour = contours[1],
            cell2_contour = contours[2],
            cell3_contour = contours[3],
         
            #
            cell_f0s = bmi_c.roi_f0s,
            cell_centres = np.int32(bmi_c.rois)[both],
            cell_ids = both,
            all_rois = np.int32(bmi_c.rois),
            low_threshold = bmi_c.low,
            high_threshold = bmi_c.high,
            low_freq = bmi_c.low_freq,
            high_freq = bmi_c.high_freq,
            cell_traces = bmi_c.roi_traces,
            sample_rate = bmi_c.sample_rate,
            post_reward_lockout = bmi_c.post_reward_lockout,
            balance_ensemble_rewards_flag = bmi_c.balance_ensemble_rewards_flag,
            rois_smooth_window = bmi_c.rois_smooth_window,
            smooth_diff_function_flag = bmi_c.smooth_diff_function_flag,
            calibration_template = bmi_c.template,
        )



len:  4


In [19]:
data = np.load('D:\bmi\DON8460\22-07-13\databmi_results.npz',allow_pickle=True)

r1 = data['rotary_encoder1_abstime']
r2 = data['rotary_encoder2_abstime']
plt.figure()
plt.plot(r1)
plt.plot(r2)
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'D:\x08mi\\DON8460\x12-07-13\\databmi_results.npz'

In [28]:
data = np.load(r'E:\bmi_backup\DON8460\22-07-13\databmi_results.npz')
ensembles = data['ensemble_activity']

temp = data['ttl_voltages']

print (ensembles.shape)
plt.figure()
plt.plot(temp)
#plt.plot(ensembles[0])
#plt.plot(ensembles[1])
plt.show()

(2, 1000)


In [35]:
#
print (gr.diff.shape)
plt.figure(0)
plt.plot(gr.diff)
plt.show()

(20000,)


In [ ]:
#######################################################
######### MANUAL ROI SELECTOR - DO NOT DELETE #########
#######################################################

# # importing the module
# import cv2

# # function to display the coordinates of
# # of the points clicked on the image
# def click_event(event, x, y, flags, params):

#     # checking for left mouse clicks
#     if event == cv2.EVENT_LBUTTONDOWN:

#         # displaying the coordinates
#         # on the Shell
#         print(x, ' ', y)
#         rois_manual.append([x,y])

#         # displaying the coordinates
#         # on the image window
#         #font = cv2.FONT_HERSHEY_SIMPLEX
#         img[y-2:y+3, x-2:x+3] = 0
   
#         #cv2.putText(img, str(x) + ',' +
#         #            str(y), (x,y), font,
#         #            .3, (255, 0, 0), 2)
#         cv2.imshow('image', img)

#     # checking for right mouse clicks	
#     #if event==cv2.EVENT_RBUTTONDOWN:
#     #    
#     #    np.save()

# # driver function
# if __name__=="__main__":
    
#     global rois_manual
    
#     rois_manual = []
    
#     # reading the image
#     #img = cv2.imread('lena.jpg', 1)
#     img = std_map.copy()
    
#     img = cv2.resize(img, (int(img.shape[0]*1.5),
#                            int(img.shape[1]*1.5))) 

#     # displaying the image
#     cv2.imshow('image', img)

#     # setting mouse handler for the image
#     # and calling the click_event() function
#     cv2.setMouseCallback('image', click_event)

#     # wait for a key to be pressed to exit
#     cv2.waitKey(0)

#     # close the window
#     cv2.destroyAllWindows()

# print (" DONE LABELING: ")
# print ("ROIS: ", rois_manual)



In [2]:
data = np.load('/media/cat/4TB/donato/BSCOPE_tests/rois_pixels_and_thresholds.npz')

low_thresh = data['low_threshold']
high_thresh = data['high_threshold']

print (low_thresh, high_thresh)



-768.7697339352011 546.5230278373468


In [10]:

octave_size = 0.25

# 
def get_octave_frequencies(low_freq,
                  high_freq,
                  octave_size):
    
    #
    octaves = []
    
    #
    octaves.append(low_freq)
    temp = low_freq
    while True:
        temp = temp * (1+octave_size)
        if temp>high_freq:
            break
        octaves.append(temp)

    return octaves
#
octaves = get_octave_frequencies(2000,
              18000,
              octave_size)
print (len(octaves),octaves)
      
    


10 [2000, 2500.0, 3125.0, 3906.25, 4882.8125, 6103.515625, 7629.39453125, 9536.7431640625, 11920.928955078125, 14901.161193847656]
